# 5G 用户预测

本 Notebook 展示课程作业的建模流程：读取数据、分析目标分布、构建两种算法、使用 AUC 进行比较。

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from src.modeling import create_models, split_features_target, load_training_data

DATA_PATH = 'train.csv'
RANDOM_STATE = 42

## 1. 读取数据

In [ ]:
df = load_training_data(DATA_PATH, max_rows=50000)
print(df.shape)
df.head()

## 2. 目标分布分析

In [ ]:
target_counts = df['target'].value_counts().sort_index()
positive_rate = df['target'].mean()
print(target_counts)
print(f'Positive rate: {positive_rate:.4%}')

正样本比例较低，因此本实验采用 AUC 作为主要评价指标，并在模型中使用类别权重处理不平衡问题。

## 3. 划分训练集与验证集

In [ ]:
X, y = split_features_target(df)
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
X_train.shape, X_valid.shape

## 4. 训练两种模型并计算 AUC

In [ ]:
from sklearn.metrics import roc_auc_score

models = create_models(random_state=RANDOM_STATE)
scores = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict_proba(X_valid)[:, 1]
    scores[name] = roc_auc_score(y_valid, pred)
scores

## 5. 当前实验结果

- Logistic Regression AUC: **0.8428**
- Random Forest AUC: **0.8716**
- 最佳模型：**random_forest**

In [ ]:
import matplotlib.pyplot as plt

pd.Series(scores).sort_values().plot(kind='barh', title='Validation AUC Comparison')
plt.xlabel('AUC')
plt.xlim(0, 1)
plt.show()

## 6. 结论

随机森林在验证集上的 AUC 更高，说明非线性集成模型更适合当前用户预测任务。后续可以使用全量数据、交叉验证和梯度提升树模型进一步提升效果。